# Projet de groupe Web scraping : prétraitement des collectes

**Cours :** Collecte et nettoyage des données (projet UA2)
**Groupe :** Yacine Lezoul (2748526) et Mouncif Bouchachi (2743259)

Deuxième notebook du projet. Le premier (`01_collecte.ipynb`) collecte les inventaires et écrit un CSV
par site et par jour dans `resultats/brut/`. Celui-ci fusionne **toutes** les collectes, les nettoie et
produit les tables d'analyse. Le code est dans `src/pretraitement.py`, les règles dans `src/config.py`.

| Section | Étape | Fonction |
|---|---|---|
| 0 | Préparation | |
| 1 | Chargement de toutes les collectes, sans perdre la source | `charger_collectes()` |
| 2 | Types : NIV, nombres, dates, prix non affichés | `nettoyer_types()` |
| 3 | Harmonisation des valeurs, complément par le NIV (API officielle vPIC) | `harmoniser()`, `completer_avec_vpic()` |
| 4 | Prix comparable entre plateformes | `ajouter_prix_comparable()` |
| 5 | Contrôles de qualité (on signale, on ne supprime pas) | `controler_qualite()` |
| 6 | Table des observations : 1 véhicule, 1 site, 1 jour | `construire_observations()` |
| 7 | Table des véhicules : 1 NIV | `construire_vehicules()` |
| 8 | Récapitulatif avant / après le prétraitement | `recapitulatif()` |
| 9 | Enregistrement et modèle pour Power BI | `enregistrer()` |

**Le modèle à deux tables**

```
concessionnaires.csv (1 ligne par site)
        │ site_id
        │ 1 ── *
        ▼
observations_quotidiennes.csv (1 ligne = 1 véhicule vu sur 1 site, 1 jour)   ← l'historique : prix et présence jour après jour
        ▲
        │ * ── 1
        │ cle_vehicule
vehicules_occasion_consolide.csv (1 ligne par véhicule)                        ← le résumé : statut, prix initial et actuel, durée en ligne
```

Le même travail se lance sans notebook : `python src/pipeline.py --etapes enrichissement pretraitement`
(et chaque nuit, après la collecte, par `scripts/collecte_quotidienne.bat`). L'étape `enrichissement`
(module `src/enrichissement.py`) décode les nouveaux NIV par l'API officielle vPIC et les garde dans un cache ;
ce notebook lit seulement ce cache et n'envoie aucune requête.

## 0. Préparation

Même cellule que dans le premier notebook : on retrouve la racine du projet, puis on importe les modules.
`importlib.reload` relit `config.py` et `pretraitement.py` si on les a modifiés sans redémarrer le noyau.

In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd


def trouver_racine(depart=None):
    """Remonte les dossiers jusqu'à trouver celui qui contient src/projet_scraping.py."""
    dossier = Path(depart or Path.cwd()).resolve()
    for candidat in [dossier, *dossier.parents]:
        if (candidat / "src" / "projet_scraping.py").exists():
            return candidat
    raise FileNotFoundError("Dossier src/ introuvable : lancez le notebook depuis le projet.")


RACINE = trouver_racine()
if str(RACINE / "src") not in sys.path:
    sys.path.insert(0, str(RACINE / "src"))

import config
import pretraitement as pt

importlib.reload(config)
importlib.reload(pt)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 60)
print("Racine du projet :", RACINE)
print("Fichiers bruts   :", len(list(config.DOSSIER_BRUT.glob("*.csv"))), "CSV dans", config.DOSSIER_BRUT.relative_to(RACINE))

## 1. Chargement de toutes les collectes

Tous les CSV de `resultats/brut/` sont lus en texte puis empilés. Chaque ligne reçoit quatre colonnes de source :

| Colonne | Contenu | Pourquoi |
|---|---|---|
| `fichier_source` | nom du CSV d'origine | retrouver la ligne brute à tout moment |
| `groupe` | groupe propriétaire du site (`config.GROUPES`) | un même véhicule peut être affiché par deux sites du même groupe |
| `jour` | date de collecte (AAAA-MM-JJ) | clé de l'historique quotidien |
| `collecte_complete` | `False` si la collecte est déclarée incomplète (`config.COLLECTES_INCOMPLETES`) | éviter les faux « nouveaux » et faux « retirés » |

Le tableau ci-dessous compte les lignes par site et par jour : c'est la preuve de la collecte quotidienne.

In [ ]:
brut = pt.charger_collectes()
brut.pivot_table(index=["plateforme", "site_id"], columns="jour", values="fichier_source",
                 aggfunc="size", fill_value=0)

In [ ]:
pd.DataFrame(config.COLLECTES_INCOMPLETES)

## 2. Types : NIV, nombres, dates, prix non affichés

- **NIV** : les fichiers Dilawri d'avant le 2026-10-23 contiennent « VIN: » devant le numéro (le site
  l'affiche dans le même élément). L'extracteur est corrigé, et le prétraitement nettoie aussi les anciens fichiers.
- **Nombres** : année et kilométrage en entiers, montants en décimaux.
- **Prix à 0** : un prix de 0 $ n'est pas un prix (« appelez-nous ») ; il devient manquant et
  `prix_non_affiche = True`, pour ne pas fausser les moyennes.

In [ ]:
df = pt.nettoyer_types(brut)
exemples = df[df["niv_brut"] != df["niv"]][["site_id", "niv_brut", "niv"]].drop_duplicates("site_id")
exemples

In [ ]:
df[df["prix_non_affiche"]].groupby(["plateforme", "site_id"]).size().rename("prix non affichés").to_frame()

## 3. Harmonisation des valeurs

Chaque plateforme écrit les mêmes informations à sa façon, en français ou en anglais (« Auto. », « Automatic »,
« Boîte automatique », « CVT »). Les règles de `config.py` sont des paires (motif, valeur) : la première qui
correspond gagne. La valeur d'origine est gardée dans une colonne `*_brut`.

Deux cas particuliers :
- **état** : chez Dilawri, la colonne contient la disponibilité (« IN-STOCK », « INCOMING VEHICLE ») et non l'état ;
  on sépare donc `etat` (Occasion, Démo) et `disponibilite` (En stock, En arrivage, Non précisé) ;
- **carburant, catégorie, traction** : quand le site ne les publie pas, on les complète par le décodage officiel
  du NIV (section 3.1 ci-dessous) ; s'il manque encore une catégorie, on reprend celle du même modèle vu sur un
  autre site (`categorie_deduite = True`). L'origine de chaque valeur est dans `carburant_source`,
  `categorie_source` et `traction_source`.

Les tableaux montrent chaque valeur d'origine et la valeur harmonisée correspondante.

In [ ]:
df = pt.harmoniser(df)

def avant_apres(colonne):
    brut_col = f"{colonne}_brut"
    return (df.groupby([brut_col, colonne], dropna=False).size()
              .rename("lignes").reset_index().sort_values([colonne, "lignes"], ascending=[True, False]))

avant_apres("transmission")

In [ ]:
avant_apres("traction")

In [ ]:
avant_apres("categorie")

In [ ]:
df.groupby(["plateforme", "etat_brut", "etat", "disponibilite"]).size().rename("lignes").reset_index()

In [ ]:
df["famille_couleur"].value_counts().rename("lignes").to_frame().T

### 3.1 Complément par le NIV : l'API officielle vPIC

Le NIV (17 caractères, norme ISO 3779) code le constructeur, le modèle, la carrosserie et le moteur. La NHTSA
(département des Transports des États-Unis) publie gratuitement, sans inscription, l'API **vPIC**, qui traduit ces
codes à partir des déclarations des constructeurs. C'est la voie « API officielle » de la partie 1 du cours,
pour cette information-là.

```
K M H D 8 4 L F 8 H U 3 4 4 3 2 3
└─┬─┘ └────┬────┘ │ │ │ └────┬────┘
 WMI     VDS      │ │ │   numéro de série
 constructeur     │ │ usine
                  │ année modèle (H = 2017)
  modèle, carrosserie, moteur
              chiffre de contrôle
```

- Le module `src/enrichissement.py` envoie les NIV par lots de 50 (maximum de la méthode `DecodeVINValuesBatch`),
  2 s entre deux lots, la nuit comme le demande la NHTSA pour les traitements par lots. Chaque NIV n'est décodé
  qu'une fois : les résultats sont gardés dans `resultats/niv_decodes_vpic.csv`.
- Le prétraitement ne complète que les **cases vides** : une valeur publiée par le concessionnaire n'est jamais remplacée.
- La **couleur** n'est pas codée dans le NIV : elle ne peut pas être complétée par cette source.
- vPIC couvre les véhicules homologués pour la vente aux États-Unis ; un NIV non reconnu reste vide, sans invention.

In [ ]:
if config.FICHIER_NIV_DECODES.exists():
    cache = pd.read_csv(config.FICHIER_NIV_DECODES, encoding="utf-8-sig", dtype=str)
    print(f"{len(cache)} NIV décodés dans le cache ; code d'erreur vPIC 0 (décodage complet) : "
          f"{(cache['vpic_code_erreur'].fillna('').str.split(',').str[0] == '0').sum()}")
else:
    print("Cache absent : lancer d'abord  python src/pipeline.py --etapes enrichissement")
pd.DataFrame({c: df[f"{c}_source"].fillna("manquant").value_counts() for c in ["carburant", "categorie", "traction"]}).fillna(0).astype(int)

In [ ]:
df.groupby(["carburant_source", "carburant"], dropna=False).size().rename("lignes").reset_index()

## 4. Prix comparable entre plateformes

Sur certains sites Dilawri, le prix final inclut des frais détaillés dans la ventilation du prix
(prix + frais - rabais = prix final, vérifié sur les 120 premiers véhicules) ; Magnetis et D2C affichent
un prix sans ces frais. Comparer directement désavantagerait ces sites. `prix_comparable = prix_final - frais`
met tout le monde sur la même base ; le prix affiché reste disponible (`prix_final`).
Le tableau montre quels sites ajoutent des frais, et combien.

In [ ]:
df = pt.ajouter_prix_comparable(df)
(df.dropna(subset=["frais"])
   .groupby(["site_id", "frais"]).size().rename("véhicules").reset_index())

## 5. Contrôles de qualité

On **signale** sans supprimer : une démo récente à 11 km est normale, une Journey 2010 à 11 km ne l'est pas,
mais seule l'analyse peut décider de l'exclure. Les seuils sont dans `config.py`.
Une valeur qu'aucune règle ne reconnaît apparaît ici : il suffit alors d'ajouter une règle dans `config.py`.

In [ ]:
df, rapport = pt.controler_qualite(df)
rapport

In [ ]:
df[df["km_suspect"] | df["niv_invalide"]][["site_id", "jour", "niv", "annee", "marque", "modele", "km",
                                            "km_suspect", "niv_invalide"]]

## 6. Table des observations : 1 véhicule, 1 site, 1 jour

C'est l'historique. Si un site a été collecté deux fois le même jour (lancement manuel + tâche de nuit),
la collecte la plus récente est gardée. La clé du véhicule est le NIV ; un véhicule sans NIV reçoit une clé
« SANS-NIV-site-annonce » pour ne pas être perdu.

In [ ]:
observations = pt.construire_observations(df)
print(observations.shape)
observations.head()

## 7. Table des véhicules : 1 NIV

Résumé de l'historique de chaque véhicule :

| Colonne | Calcul |
|---|---|
| `sites_vus`, `nb_sites`, `groupes` | tous les sites où le NIV a été vu (inventaire partagé dans un groupe) |
| `date_premiere_vue`, `date_derniere_vue`, `jours_en_ligne_min` | durée observée (au moins : il était peut-être en ligne avant le début du suivi) |
| `prix_initial`, `prix_actuel`, `variation_prix`, `nb_changements_prix` | sur `prix_comparable` ; si plusieurs sites le même jour, le plus bas |
| `statut` | « En ligne » si vu à la dernière collecte **complète** d'un de ses sites, sinon « Retiré » (vendu ou annonce retirée : impossible de savoir lequel) |
| `nouveau` | apparu après la première collecte complète de son site |

Les collectes incomplètes (Dilawri, 2026-10-09) restent dans l'historique, mais ne servent pas de repère :
sinon les véhicules manqués ce jour-là passeraient pour des « nouveaux ».

In [ ]:
vehicules = pt.construire_vehicules(observations)
print(vehicules.shape)
pd.crosstab(vehicules["concessionnaire_actuel"], vehicules["statut"], margins=True, margins_name="Total")

In [ ]:
baisses = vehicules[vehicules["variation_prix"] < 0]
hausses = vehicules[vehicules["variation_prix"] > 0]
print(f"{len(baisses)} baisses de prix (médiane {baisses['variation_prix'].median():,.0f} $), "
      f"{len(hausses)} hausses, {vehicules['nouveau'].sum()} nouveaux véhicules depuis le début du suivi")
baisses.nsmallest(10, "variation_prix")[["annee", "marque", "modele", "concessionnaire_actuel",
                                          "prix_initial", "prix_actuel", "variation_prix", "statut"]]

## 8. Récapitulatif avant / après le prétraitement

Le tableau compare les données telles que collectées (fichiers de `resultats/brut/`) aux données prétraitées :
volume, identifiant, écritures différentes d'une même information, valeurs manquantes, prix, contrôles et
traçabilité. Il est recalculé à chaque exécution et enregistré dans `resultats/recap_pretraitement.csv`.

In [ ]:
recap = pt.recapitulatif(brut, df, observations, vehicules)
pd.set_option("display.max_colwidth", 110)
recap

## 9. Enregistrement et modèle pour Power BI

Cinq fichiers dans `resultats/`, en `utf-8-sig` (lecture correcte dans Excel et Power BI) :

```
resultats/
├── observations_quotidiennes.csv     # table de faits : prix et présence jour après jour
├── vehicules_occasion_consolide.csv  # table véhicule : statut, prix initial et actuel, durée en ligne
├── concessionnaires.csv              # dimension : site, ville, province, plateforme, groupe
├── rapport_qualite.csv               # contrôles de qualité
└── recap_pretraitement.csv           # récapitulatif avant / après
```

Dans Power BI : *Obtenir les données > Texte/CSV* pour les trois tables, puis deux relations
(*Modélisation > Gérer les relations*) :
`concessionnaires[site_id]` 1 ── * `observations[site_id]` et
`vehicules[cle_vehicule]` 1 ── * `observations[cle_vehicule]`.
Un segment (slicer) sur la ville ou le groupe filtre alors les deux autres tables.

In [ ]:
concessionnaires = pt.construire_concessionnaires()
for table, chemin in [(observations, config.FICHIER_OBSERVATIONS), (vehicules, config.FICHIER_VEHICULES),
                      (concessionnaires, config.FICHIER_CONCESSIONNAIRES), (rapport, config.FICHIER_QUALITE),
                      (recap, config.FICHIER_RECAP)]:
    pt.enregistrer(table, chemin)

In [ ]:
# Vérification : relecture des fichiers écrits (encodage, nombre de lignes, clés uniques)
relu = pd.read_csv(config.FICHIER_VEHICULES, encoding="utf-8-sig")
obs_relu = pd.read_csv(config.FICHIER_OBSERVATIONS, encoding="utf-8-sig")
assert relu["cle_vehicule"].is_unique, "une clé véhicule en double"
assert not obs_relu.duplicated(["cle_vehicule", "site_id", "jour"]).any(), "une observation en double"
assert set(obs_relu["cle_vehicule"]) == set(relu["cle_vehicule"]), "clés différentes entre les deux tables"
print(f"OK : {len(obs_relu)} observations, {len(relu)} véhicules, clés cohérentes")